<a href="https://colab.research.google.com/github/alvaradodiegom/marketing-analytics-diego-alvarado/blob/main/Semana_9/Laboratorio_Agente_Churn_Uplift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio · Amplía tu agente: churn y uplift xb
**Marketing Analytics · Universidad Rafael Landívarx**

En la demo construimos un agente con **una** herramienta (`calcular_clv`). Hoy le van a enseñar **dos herramientas más** del toolkit del curso, paso a paso, igual que en la demo.

| Parte | Herramienta | Pregunta de negocio | Tiempo |
|---|---|---|---|
| 0 | Preparación (`calcular_clv` + el ciclo del agente) | Lo de la demo, ya listo | 10 min |
| 1 | `churn_maximo_tolerable` | Si subo el precio, ¿cuánto churn aguanto sin perder valor? | 30 min |
| 2 | `valor_campana_retencion` | ¿Vale la pena esta campaña de retención? | 30 min |
| 3 | Evaluación final | ¿El agente acierta y no cae en trampas? | 15 min |

**Cada herramienta sigue los mismos 6 pasos:**
1. **Entender** la pregunta y la fórmula
2. **Resolver a mano** un ejemplo (para tener con qué comparar)
3. **Escribir** la herramienta (llenando los espacios `___`)
4. **Probarla sola**: ¿da lo mismo que a mano?
5. **Dársela al agente** y ver si la usa (líneas "Paso 1: …")
6. **Trampa**: ¿el agente se deja engañar?

> **Cómo leer las celdas:** las que tienen `___` las completan ustedes. Si una celda da `NameError: name '___' is not defined`, es que falta llenar un espacio.

---
# Parte 0 · Preparación



### Paso 0.0 · Crear su llave de Gemini (API key)
La API key (Application Programming Interface key) es la **clave que deja a su código hablar con Gemini**. Es gratis y se crea una sola vez.

1. Entren a **[aistudio.google.com](https://aistudio.google.com)** e inicien sesión con su cuenta de Google. Si su cuenta de la universidad no los deja entrar, usen una cuenta personal de Gmail.
2. Acepten los términos si es la primera vez.
3. Abajo a la izquierda, hagan clic en el **ícono de llave (Get API key)**.



4. Hagan clic en **Create API key** (Crear clave de API). Si les pide un proyecto, elijan el que aparece o creen uno nuevo; el nombre no importa.
5. Abran la llave y usen el botón **Copiar clave**. Las llaves nuevas empiezan con `AQ.` y las antiguas con `AIza`; las dos sirven.



6. **Guárdenla fuera del código:**
   - **VS Code:** creen un archivo llamado exactamente `.env` (con punto al inicio y sin `.txt`) en la misma carpeta del notebook, con una sola línea, sin comillas ni espacios: `GEMINI_API_KEY=AQ.su_llave`
   - **Colab:** menú izquierdo → ícono de llave (Secrets) → **Add new secret** → Name: `GEMINI_API_KEY`, Value: su llave, y activen **Notebook access**.

> **Nunca** peguen la llave dentro del código, en un chat o en GitHub (agreguen `.env` al `.gitignore`). Si se les escapa, bórrenla en AI Studio (**Borrar clave**) y creen otra.
> En Colab, `userdata.get("GEMINI_API_KEY")` recibe el **nombre** del secreto, no la llave.
> El plan gratuito permite unas **15 llamadas por minuto**: si ven el error 429, esperen un minuto.


### Paso 0.1 · Conectar con Gemini
Igual que en la demo: lee la llave (en Colab desde Secrets, en VS Code desde el archivo `.env`) y elige un modelo "flash-lite".

- **VS Code:** en la terminal `pip install google-genai python-dotenv ipykernel` y el archivo `.env` con `GEMINI_API_KEY=su_llave`
- **Colab:** `!pip -q install google-genai` y la llave en Secrets (ícono de llave) con el nombre `GEMINI_API_KEY`

**Qué deben ver:** `Modelo: gemini-…-flash-lite`

In [1]:
import os, json, re, time
from google import genai
from google.genai import types, errors

try:
    from google.colab import userdata          # Colab: llave en Secrets
    API_KEY = userdata.get("GEMINI_API_KEY")
except ImportError:                            # VS Code: llave en el archivo .env
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)

# Elegir un modelo "flash-lite" disponible en su cuenta
excluir = ("tts", "image", "live", "audio", "embedding", "preview", "exp")
modelos = []
for m in client.models.list():
    nombre = m.name.replace("models/", "")
    if "generateContent" in (m.supported_actions or []) and "gemini" in nombre:
        if not any(x in nombre for x in excluir):
            modelos.append(nombre)
lite = sorted(m for m in modelos if "flash-lite" in m)
MODELO = lite[-1] if lite else sorted(modelos)[-1]
print("Modelo:", MODELO)

D_MENSUAL = 0.008    # tasa de descuento mensual del curso (0.8%)

Modelo: gemini-flash-lite-latest


### Paso 0.2 · La herramienta de la demo: `calcular_clv`
Ya viene hecha. Fíjense en sus tres partes, porque las de hoy se escriben igual:
1. **Docstring** (texto entre comillas triples): es lo **único** que el LLM lee de la herramienta.
2. **Barandas** (`if … return {"error": …}`): revisan los datos antes de calcular.
3. **Cálculo** y `return` con un diccionario.

**Qué deben ver:** `{'clv': 500.0}`

In [2]:
def calcular_clv(margen_mensual: float, churn_mensual: float) -> dict:
    """Calcula el CLV (Customer Lifetime Value, valor de vida del cliente) con la fórmula
    del curso: CLV = m × (1 − c) ÷ (c + d), con d = 0.008 (0.8% mensual).
    margen_mensual: margen por cliente al mes en US$.
    churn_mensual: proporción de clientes que se va cada mes (0.04 = 4%)."""
    # Baranda: el churn tiene que estar entre 0% y 100%
    if churn_mensual <= 0 or churn_mensual >= 1:
        return {"error": "El churn debe estar entre 0 y 1 (0% y 100%). Si dijeron 4, pregunta si es 4% (0.04)."}
    # Baranda: el margen tiene que ser positivo
    if margen_mensual <= 0:
        return {"error": "El margen debe ser mayor que 0."}
    # Cálculo
    m, c, d = margen_mensual, churn_mensual, D_MENSUAL
    return {"clv": round(m * (1 - c) / (c + d), 2)}

print(calcular_clv(25, 0.04))     # debe dar 500.0

{'clv': 500.0}


### Paso 0.3 · El ciclo del agente
Es el mismo de la demo, con una diferencia: usa el menú `HERRAMIENTAS`, así que cada herramienta nueva solo hay que **agregarla al menú** (lo harán en el paso 5 de cada parte).

- `llamar_gemini`: si Gemini dice "te pasaste del límite" (error 429, 15 llamadas por minuto), espera 40 segundos y reintenta.
- `preguntar`: 1) el LLM piensa → 2) si no pidió herramientas, terminó → 3) ejecutamos lo que pidió → 4) le devolvemos el resultado y repite.

**Qué deben ver:** una línea `Paso 1: calcular_clv(...)` y una explicación que diga US$500.

In [3]:
HERRAMIENTAS = {"calcular_clv": calcular_clv}     # el menú: nombre → función

INSTRUCCIONES = """Eres un tutor experto en Marketing Analytics para estudiantes de maestría.
1. Para cualquier número, usa una herramienta. Nunca calcules de memoria.
2. Si falta un dato, pregunta; no lo supongas.
3. Si una herramienta devuelve un error, explícaselo al usuario y pide el dato correcto.
4. Explica en pasos cortos, muestra la fórmula y define cada sigla."""


def llamar_gemini(contenido, config=None):
    """Llama a Gemini. Si nos pasamos del límite gratuito (error 429), espera 40 segundos y reintenta."""
    for intento in range(4):
        try:
            return client.models.generate_content(model=MODELO, contents=contenido, config=config)
        except errors.ClientError as e:
            if e.code == 429 and intento < 3:
                print("Límite de Gemini (15 llamadas por minuto). Espero 40 segundos…")
                time.sleep(40)
            else:
                raise


def preguntar(pregunta, historial=None, max_pasos=8):
    """Le hace una pregunta al agente con TODAS las herramientas del menú. Devuelve (respuesta, historial)."""
    config = types.GenerateContentConfig(
        system_instruction=INSTRUCCIONES,
        tools=list(HERRAMIENTAS.values()),
        temperature=0.2,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
    historial = list(historial or [])
    historial.append(types.Content(role="user", parts=[types.Part.from_text(text=pregunta)]))

    for paso in range(1, max_pasos + 1):
        r = llamar_gemini(historial, config)               # 1. el LLM piensa
        historial.append(r.candidates[0].content)
        if not r.function_calls:                            # 2. sin herramientas = terminó
            return r.text, historial
        resultados = []
        for pedido in r.function_calls:                     # 3. ejecutamos lo que pidió
            datos = dict(pedido.args)
            try:
                salida = HERRAMIENTAS[pedido.name](**datos)
            except Exception as e:                          # baranda: no se cae, le avisa
                salida = {"error": str(e)}
            print(f"Paso {paso}: {pedido.name}({datos}) → {salida}")
            resultados.append(types.Part.from_function_response(name=pedido.name,
                                                                response={"resultado": salida}))
        historial.append(types.Content(role="user", parts=resultados))   # 4. le devolvemos todo
    return "Me detuve: llegué al máximo de pasos.", historial


def revisar(descripcion, obtenido, esperado, tolerancia=0.01):
    """Compara el resultado de una herramienta con el ejemplo resuelto a mano."""
    if obtenido is not None and abs(obtenido - esperado) <= tolerancia * max(1, abs(esperado)):
        print("OK      ", descripcion, "→", obtenido)
    else:
        print("REVISAR ", descripcion, "→ obtuvo", obtenido, "· esperado", esperado)


respuesta, h = preguntar("Margen US$25 y churn 4% mensual. ¿Cuál es el CLV?")
print("\n" + respuesta)

Paso 1: calcular_clv({'margen_mensual': 25, 'churn_mensual': 0.04}) → {'clv': 500.0}

El **CLV (Customer Lifetime Value)**, o valor de vida del cliente, es de **US$ 500**.

Aquí te explico el paso a paso:

### 1. Fórmula utilizada
$$\text{CLV} = \frac{m \times (1 - c)}{c + d}$$

### 2. Definición de las variables y datos
* **$m$ (Margen mensual):** US$ 25 (es el beneficio bruto que genera un cliente al mes).
* **$c$ (Churn mensual):** $0.04$ o $4\%$ (es la tasa de abandono de clientes cada mes).
* **$d$ (Tasa de descuento mensual):** $0.008$ o $0.8\%$ (es la tasa fija establecida en el curso para traer los flujos futuros a valor presente).

### 3. Sustitución y cálculo
$$\text{CLV} = \frac{25 \times (1 - 0.04)}{0.04 + 0.008}$$

$$\text{CLV} = \frac{25 \times 0.96}{0.048}$$

$$\text{CLV} = \frac{24}{0.048} = 500$$


---
# Parte 1 · `churn_maximo_tolerable`

### Paso 1.1 · Entender la pregunta
**Situación:** queremos subir el precio. El margen por cliente sube, pero algunos clientes se van a ir.
**Pregunta:** ¿cuál es el churn máximo que aguantamos **sin que baje el CLV**?

**Fórmula del curso:**

$$c' = \frac{m' - CLV \times d}{m' + CLV}$$

| Símbolo | Qué es | Ejemplo |
|---|---|---|
| c' | churn mensual **máximo tolerable** (lo que buscamos) | ? |
| m' | margen mensual **nuevo** por cliente, con el precio nuevo (US$) | 28 |
| CLV | valor de vida del cliente **de hoy** (US$) | 500 |
| d | tasa de descuento mensual del curso | 0.008 |

**Lectura:** si después del aumento el churn sube **más** que c', el aumento de precio destruye valor.

### Paso 1.2 · Resolver a mano
Hoy el CLV es **US$500** y con el precio nuevo el margen pasa a **US$28**.

| Paso | Cuenta | Resultado |
|---|---|---|
| Arriba | 28 − 500 × 0.008 = 28 − 4 | **24** |
| Abajo | 28 + 500 | **528** |
| c' | 24 ÷ 528 | **0.0455 = 4.55%** |

**Comprobación:** con m = 28 y c = 0.0455, `calcular_clv` debe dar ≈ US$500 (el mismo CLV de antes). La celda de abajo hace las cuentas paso a paso, como una calculadora:

In [4]:
margen_nuevo, clv_actual = 28, 500

arriba = margen_nuevo - clv_actual * D_MENSUAL
abajo = margen_nuevo + clv_actual
c_max = arriba / abajo
print("Arriba:", arriba, " · Abajo:", abajo, " · c' =", round(c_max, 4), f"({c_max:.2%})")

# Comprobación: con ese churn, el CLV nuevo debe ser igual al de hoy
print("CLV con el precio nuevo y c':", calcular_clv(margen_nuevo, c_max))

Arriba: 24.0  · Abajo: 528  · c' = 0.0455 (4.55%)
CLV con el precio nuevo y c': {'clv': 500.0}


### Paso 1.3 · Escribir la herramienta
Completen los `___`. Guíense por la celda de arriba: es la misma cuenta, ahora dentro de una función.

**Las dos barandas:**
1. El margen nuevo y el CLV deben ser **positivos**.
2. Si el margen nuevo es menor que CLV × 0.008, `arriba` sale negativo: **ningún churn** mantiene el CLV. Hay que devolver un error que lo explique.

**Qué deben ver:** `{'churn_maximo': 0.0455}`

In [5]:
def churn_maximo_tolerable(margen_nuevo: float, clv_actual: float) -> dict:
    """Churn mensual máximo que puede tener un segmento después de un cambio de precio
    sin que su CLV baje: c' = (m' − CLV × d) ÷ (m' + CLV), con d = 0.008.
    margen_nuevo: margen mensual por cliente con el precio nuevo (US$).
    clv_actual: CLV de hoy del segmento (US$)."""
    # Baranda 1: los dos datos tienen que ser positivos
    if margen_nuevo <= 0 or clv_actual <= 0:
        return {"error": "El margen nuevo y el CLV actual deben ser mayores que 0."}

    # Cálculo (igual que en el paso 1.2)
    arriba = margen_nuevo - clv_actual * D_MENSUAL   # pista: margen_nuevo − clv_actual × D_MENSUAL
    abajo = margen_nuevo + clv_actual   # pista: margen_nuevo + clv_actual
    c = arriba / abajo

    # Baranda 2: si c sale negativo, ningún churn mantiene el CLV
    if c < 0:                          # pista: c < 0
        return {"error": "Con ese margen nuevo ningún churn mantiene el CLV: "
                         "el margen es menor que CLV × 0.008."}

    return {"churn_maximo": round(c, 4)}

print(churn_maximo_tolerable(28, 500))    # esperado: {'churn_maximo': 0.0455}

{'churn_maximo': 0.0455}


<details>
<summary><b>¿Se trabaron? Abran aquí la solución</b> (inténtenlo primero solos)</summary>

```python
def churn_maximo_tolerable(margen_nuevo: float, clv_actual: float) -> dict:
    """Churn mensual máximo que puede tener un segmento después de un cambio de precio
    sin que su CLV baje: c' = (m' − CLV × d) ÷ (m' + CLV), con d = 0.008.
    margen_nuevo: margen mensual por cliente con el precio nuevo (US$).
    clv_actual: CLV de hoy del segmento (US$)."""
    if margen_nuevo <= 0 or clv_actual <= 0:
        return {"error": "El margen nuevo y el CLV actual deben ser mayores que 0."}
    arriba = margen_nuevo - clv_actual * D_MENSUAL
    abajo = margen_nuevo + clv_actual
    c = arriba / abajo
    if c < 0:
        return {"error": "Con ese margen nuevo ningún churn mantiene el CLV: "
                         "el margen es menor que CLV × 0.008."}
    return {"churn_maximo": round(c, 4)}
```
</details>

### Paso 1.4 · Probarla sola
Dos ejemplos resueltos a mano y una prueba de la baranda. **Las dos primeras líneas deben salir OK** antes de seguir.

Segundo ejemplo a mano: margen nuevo US$10 y CLV US$200 → (10 − 1.6) ÷ 210 = **0.04**.

In [6]:
revisar("Margen nuevo 28, CLV 500", churn_maximo_tolerable(28, 500).get("churn_maximo"), 0.0455)
revisar("Margen nuevo 10, CLV 200", churn_maximo_tolerable(10, 200).get("churn_maximo"), 0.04)

# La baranda: margen nuevo de 3 con CLV de 500 → 3 − 4 es negativo → debe devolver un error
print(churn_maximo_tolerable(3, 500))

OK       Margen nuevo 28, CLV 500 → 0.0455
OK       Margen nuevo 10, CLV 200 → 0.04
{'error': 'Con ese margen nuevo ningún churn mantiene el CLV: el margen es menor que CLV × 0.008.'}


### Paso 1.5 · Dársela al agente
Una sola línea: agregarla al menú. A partir de aquí, `preguntar` le ofrece las dos herramientas al LLM y él elige cuál usar.

**Qué deben ver:** una línea `Paso 1: churn_maximo_tolerable({'margen_nuevo': 28, 'clv_actual': 500})` y una respuesta con **4.55%**.
Si en vez de eso contesta sin línea "Paso 1: …", hizo la cuenta él solo: repitan la celda y anótenlo (el prompt solo pide; el código obliga).

In [7]:
HERRAMIENTAS["churn_maximo_tolerable"] = churn_maximo_tolerable

respuesta, h = preguntar("Hoy mi CLV es US$500. Si subo el precio y el margen pasa a US$28 al mes, "
                         "¿cuál es el churn máximo que aguanto?")
print("\n" + respuesta)

Paso 1: churn_maximo_tolerable({'clv_actual': 500, 'margen_nuevo': 28}) → {'churn_maximo': 0.0455}

El **churn máximo** que puedes tolerar sin que tu CLV baje es **4.55%** mensual (o $0.0455$ en proporción).

---

### ¿Cómo se calcula?

Usamos la fórmula para calcular el churn máximo tolerable ($c'$):

$$c' = \frac{m' - (\text{CLV} \times d)}{m' + \text{CLV}}$$

**Definición de siglas y variables:**
* **$c'$**: Churn mensual máximo tolerable (la tasa de cancelación límite después del cambio de precio).
* **$m'$**: Margen mensual nuevo por cliente. En tu caso, **US$ 28**.
* **$\text{CLV}$**: Customer Lifetime Value actual (Valor de vida del cliente). En tu caso, **US$ 500**.
* **$d$**: Tasa de descuento mensual fija (fijada en el curso en $0.008$ o **0.8%**).

---

### Paso a paso:

1. **Multiplicamos el CLV actual por la tasa de descuento ($d$):**
   $$\text{US\$ } 500 \times 0.008 = \text{US\$ } 4$$

2. **Calculamos el numerador (restando ese valor al nuevo margen):**
   $$m' - (\text

### Paso 1.6 · Trampa: ¿se deja engañar?
Le damos un margen nuevo tan bajo que la fórmula da negativo.

**Qué debe pasar:** la línea "Paso 1: …" muestra el **error de la baranda** y el agente explica que con ese margen ningún churn mantiene el CLV. **No** debe inventar un porcentaje.

In [8]:
respuesta, h = preguntar("Mi CLV es US$500 y el margen nuevo será US$3 al mes. ¿Cuánto churn aguanto?")
print("\n" + respuesta)

Paso 1: churn_maximo_tolerable({'clv_actual': 500, 'margen_nuevo': 3}) → {'error': 'Con ese margen nuevo ningún churn mantiene el CLV: el margen es menor que CLV × 0.008.'}

El cálculo ha devuelto el siguiente error: el margen nuevo es tan bajo que, para mantener un CLV de US$500, necesitarías un *churn* negativo (es decir, que los clientes no solo no se fueran, sino que ingresaran espontáneamente), lo cual no es posible en la práctica.

Te explico por qué ocurre esto paso a paso con la fórmula utilizada:

### 1. Fórmula del Churn Máximo Tolerable
La fórmula para calcular el *churn* máximo ($c'$) tras un cambio de precio es:

$$c' = \frac{m' - (CLV \times d)}{m' + CLV}$$

### 2. Definición de siglas y variables
* **$c'$**: *Churn* mensual máximo tolerable (la proporción de clientes que se puede ir cada mes sin que el CLV disminuya).
* **$m'$**: Margen mensual nuevo por cliente. En tu caso: **US$3**.
* **$CLV$**: *Customer Lifetime Value* actual (Valor de vida del cliente). En tu caso: 

---
# Parte 2 · `valor_campana_retencion`

### Paso 2.1 · Entender la pregunta
**Situación:** queremos contactar clientes con una oferta para que no se vayan. Cada contacto cuesta.
**Pregunta:** ¿la campaña **gana o pierde** dinero?

**La clave (Actividad 2):** el **uplift** es el efecto **incremental**: cuántos clientes se quedan **gracias** a la campaña, comparando contra un **grupo de control** que no recibió nada. **No** es "el porcentaje que se quedó", porque muchos se habrían quedado igual.

| Concepto | Fórmula |
|---|---|
| Clientes retenidos extra | clientes contactados × uplift |
| Valor generado | retenidos extra × CLV |
| Costo total | clientes contactados × costo por contacto |
| **Beneficio neto** | valor generado − costo total |
| Uplift mínimo para no perder | costo por contacto ÷ CLV |

### Paso 2.2 · Resolver a mano
**1,000** clientes, uplift **5%**, CLV **US$500**, **US$10** por contacto.

| Concepto | Cuenta | Resultado |
|---|---|---|
| Retenidos extra | 1,000 × 0.05 | **50** |
| Valor generado | 50 × 500 | **US$25,000** |
| Costo total | 1,000 × 10 | **US$10,000** |
| Beneficio neto | 25,000 − 10,000 | **US$15,000 → conviene** |
| Uplift mínimo | 10 ÷ 500 | **0.02 = 2%** |

In [9]:
contactados, uplift, clv, costo = 1000, 0.05, 500, 10

retenidos = contactados * uplift
valor = retenidos * clv
costo_total = contactados * costo
beneficio = valor - costo_total
print("Retenidos extra:", retenidos, "· Valor:", valor, "· Costo:", costo_total, "· Beneficio neto:", beneficio)
print("Uplift mínimo para no perder:", costo / clv)

Retenidos extra: 50.0 · Valor: 25000.0 · Costo: 10000 · Beneficio neto: 15000.0
Uplift mínimo para no perder: 0.02


### Paso 2.3 · Escribir la herramienta
Completen los `___`, guiándose por la celda de arriba.

**Las barandas:**
1. El uplift va entre 0 y 1. Si alguien escribe 5, el agente debe preguntar si es 5% (0.05).
2. Clientes y CLV deben ser positivos; el costo no puede ser negativo.

**Qué deben ver:** un diccionario con `'beneficio_neto': 15000.0` y `'conviene': True`

In [15]:
def valor_campana_retencion(clientes_contactados: int, uplift: float,
                            clv: float, costo_por_contacto: float) -> dict:
    """Valor esperado de una campaña de retención usando uplift (efecto incremental:
    proporción de contactados que se queda GRACIAS a la campaña, medido contra un grupo de control).
    uplift = tasa de retención con campaña − tasa de retención del grupo de control (sin campaña).
    NUNCA uses la tasa de retención ("el 30% se quedó") como uplift. Si el usuario no da
    la tasa del grupo de control, NO llames esta herramienta: pídele ese dato.
    clientes_contactados: número de clientes a contactar.
    uplift: proporción retenida por la campaña (0.05 = 5%).
    clv: valor de vida de un cliente retenido (US$).
    costo_por_contacto: costo de contactar a un cliente (US$)."""

    # se cambio el texto porque cayo en la tranmpa en el paso 2.6

    # Baranda 1: el uplift es una proporción entre 0 y 1
    if uplift < 0 or uplift > 1:                          # pista: uplift < 0 or uplift > 1
        return {"error": "El uplift debe estar entre 0 y 1. Si dijeron 5, pregunta si es 5% (0.05)."}
    # Baranda 2: datos con sentido
    if clientes_contactados <= 0 or clv <= 0 or costo_por_contacto < 0:
        return {"error": "Clientes y CLV deben ser mayores que 0, y el costo no puede ser negativo."}

    # Cálculo (igual que en el paso 2.2)
    retenidos = clientes_contactados * uplift                  # pista: clientes_contactados × uplift
    valor = retenidos * clv                      # pista: retenidos × clv
    costo_total = clientes_contactados * costo_por_contacto                # pista: clientes_contactados × costo_por_contacto
    beneficio = valor - costo_total

    return {"clientes_retenidos_extra": round(retenidos, 1),
            "valor_generado": round(valor, 2),
            "costo_total": round(costo_total, 2),
            "beneficio_neto": round(beneficio, 2),
            "conviene": beneficio > 0,
            "uplift_minimo_para_no_perder": round(costo_por_contacto / clv, 4)}

print(valor_campana_retencion(1000, 0.05, 500, 10))   # esperado: beneficio_neto 15000.0, conviene True

{'clientes_retenidos_extra': 50.0, 'valor_generado': 25000.0, 'costo_total': 10000, 'beneficio_neto': 15000.0, 'conviene': True, 'uplift_minimo_para_no_perder': 0.02}


<details>
<summary><b>¿Se trabaron? Abran aquí la solución</b> (inténtenlo primero solos)</summary>

```python
def valor_campana_retencion(clientes_contactados: int, uplift: float,
                            clv: float, costo_por_contacto: float) -> dict:
    """Valor esperado de una campaña de retención usando uplift (efecto incremental:
    proporción de contactados que se queda GRACIAS a la campaña, medido contra un grupo de control).
    clientes_contactados: número de clientes a contactar.
    uplift: proporción retenida por la campaña (0.05 = 5%).
    clv: valor de vida de un cliente retenido (US$).
    costo_por_contacto: costo de contactar a un cliente (US$)."""
    if uplift < 0 or uplift > 1:
        return {"error": "El uplift debe estar entre 0 y 1. Si dijeron 5, pregunta si es 5% (0.05)."}
    if clientes_contactados <= 0 or clv <= 0 or costo_por_contacto < 0:
        return {"error": "Clientes y CLV deben ser mayores que 0, y el costo no puede ser negativo."}
    retenidos = clientes_contactados * uplift
    valor = retenidos * clv
    costo_total = clientes_contactados * costo_por_contacto
    beneficio = valor - costo_total
    return {"clientes_retenidos_extra": round(retenidos, 1),
            "valor_generado": round(valor, 2),
            "costo_total": round(costo_total, 2),
            "beneficio_neto": round(beneficio, 2),
            "conviene": beneficio > 0,
            "uplift_minimo_para_no_perder": round(costo_por_contacto / clv, 4)}
```
</details>

### Paso 2.4 · Probarla sola
Segundo ejemplo a mano: **2,000** clientes, uplift **1%**, CLV **US$300**, **US$5** por contacto → retenidos 20, valor US$6,000, costo US$10,000 → beneficio **−US$4,000** (no conviene; el uplift mínimo era 1.67%).

In [12]:
revisar("1000 clientes, 5%, CLV 500, costo 10", valor_campana_retencion(1000, 0.05, 500, 10).get("beneficio_neto"), 15000)
revisar("2000 clientes, 1%, CLV 300, costo 5", valor_campana_retencion(2000, 0.01, 300, 5).get("beneficio_neto"), -4000)
revisar("Uplift mínimo del ejemplo 1", valor_campana_retencion(1000, 0.05, 500, 10).get("uplift_minimo_para_no_perder"), 0.02)

# La baranda: un uplift de 5 (en vez de 0.05) debe devolver un error
print(valor_campana_retencion(1000, 5, 500, 10))

OK       1000 clientes, 5%, CLV 500, costo 10 → 15000.0
OK       2000 clientes, 1%, CLV 300, costo 5 → -4000.0
OK       Uplift mínimo del ejemplo 1 → 0.02
{'error': 'El uplift debe estar entre 0 y 1. Si dijeron 5, pregunta si es 5% (0.05).'}


### Paso 2.5 · Dársela al agente
**Qué deben ver:** `Paso 1: valor_campana_retencion(...)` y una respuesta con **US$15,000** de beneficio neto.

In [19]:
HERRAMIENTAS["valor_campana_retencion"] = valor_campana_retencion

respuesta, h = preguntar("Voy a contactar a 1,000 clientes. El uplift es 5%, el CLV es US$500 "
                         "y cada contacto cuesta US$10. ¿Conviene la campaña?")
print("\n" + respuesta)

Paso 1: valor_campana_retencion({'costo_por_contacto': 10, 'clientes_contactados': 1000, 'uplift': 0.05, 'clv': 500}) → {'clientes_retenidos_extra': 50.0, 'valor_generado': 25000.0, 'costo_total': 10000, 'beneficio_neto': 15000.0, 'conviene': True, 'uplift_minimo_para_no_perder': 0.02}

¡Sí conviene! Vamos a desglosarlo paso a paso utilizando la fórmula del curso de Marketing Analytics para evaluar campañas de retención por *uplift*.

---

### 1. Definición de conceptos y fórmula

* **CLV (*Customer Lifetime Value*):** Es el valor de vida del cliente (US$ 500). Representa los ingresos netos futuros proyectados que aporta un cliente durante su relación con la empresa.
* **Uplift:** Es el efecto incremental (0.05 o 5%). Es la proporción de clientes que **se queda gracias a la campaña**, medido descontando al grupo de control (es el porcentaje adicional que retienes por la intervención, no la retención total).
* **Clientes contactados:** Número total de clientes a los que se les aplicará 

### Paso 2.6 · Trampa: "el 30% se quedó"
Le damos una tasa de retención **sin grupo de control**. Eso **no** es uplift: no sabemos cuántos se habrían quedado sin la campaña.

**Qué debe pasar:** el agente explica que 30% no es uplift y **pide la tasa del grupo de control**. Si usa 0.30 como uplift y calcula, **cayó en la trampa**: mejoren el docstring (por ejemplo, agregando "nunca uses la tasa de retención como uplift") y vuelvan a correr los pasos 2.3 y 2.5.

In [18]:
respuesta, h = preguntar("Contacté a 1,000 clientes y el 30% se quedó. Con un CLV de US$500 "
                         "y US$10 por contacto, ¿cuánto ganó la campaña?")
print("\n" + respuesta)


Para calcular el valor esperado de la campaña de retención, necesitamos usar el concepto de **uplift** (efecto incremental), que mide cuántos clientes se quedaron *gracias* a la campaña, comparado con un grupo de control (clientes que no recibieron la campaña pero tenían las mismas características).

Me diste que el **30% se quedó**, pero esa es la tasa de retención total con la campaña, **no** el uplift. 

Para poder usar la herramienta y calcular el resultado correctamente, por favor dime:
* **¿Cuál fue la tasa de retención del grupo de control (sin campaña)?**


---
# Parte 3 · Evaluación final: ¿el agente acierta y no cae en trampas?
Corremos todas las preguntas de una vez.
- Las que tienen `esperado` se califican solas: el número debe aparecer en la respuesta.
- Las **trampas** (`esperado = None`) se revisan **a ojo** contra la columna "debe".

**Tarea:** agreguen al final de `BANCO` **1 trampa propia** pensada para sus herramientas (ideas: un uplift de 5 en lugar de 0.05, un margen nuevo menor al de hoy, una pregunta que mezcle meses y años).

> Son ~10 preguntas y cada una usa 2 o 3 llamadas a Gemini: es normal ver "Espero 40 segundos" un par de veces.

In [22]:
BANCO = [
    {"pregunta": "Margen US$25 y churn 4% mensual. ¿Cuál es el CLV?", "esperado": [500]},
    {"pregunta": "Margen US$12 y churn 3% mensual. ¿Cuál es el CLV?", "esperado": [306.32]},
    {"pregunta": "Hoy el CLV es US$500. Si el margen nuevo es US$28 al mes, ¿cuál es el churn máximo que aguanto?",
     "esperado": [4.55, 0.0455]},
    {"pregunta": "Contacto a 1,000 clientes, el uplift es 5%, el CLV es US$500 y cada contacto cuesta US$10. "
                 "¿Cuál es el beneficio neto?", "esperado": [15000]},
    {"pregunta": "Contacto a 2,000 clientes, uplift 1%, CLV US$300 y costo de US$5 por contacto. ¿Conviene?",
     "esperado": [-4000, 4000]},
    # --- Trampas (se revisan a ojo) ---
    {"pregunta": "¿Cuánto vale un cliente si el churn es 4%?", "esperado": None,
     "debe": "Preguntar el margen"},
    {"pregunta": "Margen US$25 y churn de 120%. ¿CLV?", "esperado": None,
     "debe": "Decir que es imposible (churn mayor a 100%)"},
    {"pregunta": "Contacté a 1,000 clientes y el 30% se quedó. Con CLV de US$500, ¿cuánto ganó la campaña?",
     "esperado": None, "debe": "Explicar que 30% no es uplift y pedir el grupo de control"},
    # --- Su trampa propia Uplift de 5 en vez de 0.05
    # {"pregunta": "...", "esperado": None, "debe": "..."},
     {"pregunta": "Contacto a 1,000 clientes, el uplift es 5, el CLV es US$500 y cada contacto cuesta US$10. ¿Conviene?",
     "esperado": None, "debe": "Preguntar si quiso decir 5% (0.05), no calcular con 5"},
]


def numeros_en(texto):
    """Saca todos los números de un texto (por ejemplo "US$15,000" → 15000.0)."""
    encontrados = []
    for n in re.findall(r"-?\d[\d,]*\.?\d*", texto or ""):
        try:
            encontrados.append(float(n.replace(",", "").rstrip(".")))
        except ValueError:
            pass
    return encontrados


def evaluar(banco=BANCO, tolerancia=0.01):
    aciertos, calificadas = 0, 0
    for i, caso in enumerate(banco, 1):
        respuesta, _ = preguntar(caso["pregunta"])
        if caso["esperado"] is None:
            print(f"\nTRAMPA {i}. — debe: {caso['debe']}\n   Respuesta: {respuesta[:300]}")
            continue
        calificadas += 1
        ok = False
        for esperado in caso["esperado"]:
            for n in numeros_en(respuesta):
                if abs(n - esperado) <= tolerancia * max(1, abs(esperado)):
                    ok = True
        aciertos += ok
        print(f"\n{'OK' if ok else 'REVISAR'} {i}. {caso['pregunta']}\n   esperado {caso['esperado']}")
    print(f"\nResultado: {aciertos} de {calificadas} preguntas con respuesta conocida")

evaluar()

Paso 1: calcular_clv({'margen_mensual': 25, 'churn_mensual': 0.04}) → {'clv': 500.0}

OK 1. Margen US$25 y churn 4% mensual. ¿Cuál es el CLV?
   esperado [500]
Paso 1: calcular_clv({'churn_mensual': 0.03, 'margen_mensual': 12}) → {'clv': 306.32}

OK 2. Margen US$12 y churn 3% mensual. ¿Cuál es el CLV?
   esperado [306.32]
Paso 1: churn_maximo_tolerable({'margen_nuevo': 28, 'clv_actual': 500}) → {'churn_maximo': 0.0455}

OK 3. Hoy el CLV es US$500. Si el margen nuevo es US$28 al mes, ¿cuál es el churn máximo que aguanto?
   esperado [4.55, 0.0455]
Paso 1: valor_campana_retencion({'clv': 500, 'uplift': 0.05, 'clientes_contactados': 1000, 'costo_por_contacto': 10}) → {'clientes_retenidos_extra': 50.0, 'valor_generado': 25000.0, 'costo_total': 10000, 'beneficio_neto': 15000.0, 'conviene': True, 'uplift_minimo_para_no_perder': 0.02}

REVISAR 4. Contacto a 1,000 clientes, el uplift es 5%, el CLV es US$500 y cada contacto cuesta US$10. ¿Cuál es el beneficio neto?
   esperado [15000]
Paso 1: v

---
# Entregable del avance (súbanlo a su rama de GitHub)
- [ ] Parte 1: `churn_maximo_tolerable` con los pasos 1.4 en OK y la trampa 1.6 probada
- [ ] Parte 2: `valor_campana_retencion` con los pasos 2.4 en OK y la trampa 2.6 probada
- [ ] Parte 3: el resultado de `evaluar()` (cuántas acierta y qué pasó con cada trampa) y **1 trampa propia** en el `BANCO`
- [ ] Una celda de texto al final: **¿en qué trampa cayó el agente y qué cambiaron para arreglarlo?**

**Notas del grupo:**

*¿en qué trampa cayó el agente y qué cambiaron para arreglarlo?*

Paso 2.6 en la Trampa: "el 30% se quedó" el agente uso el 30% que es de la retencion como si fuera el valor de uplif, y calculo la ganancia ademas respondio"que el valor estaba dentro del rango medio contra un grupo de control".


Para que el modelo no cayera en la trampa se agrego en el paso 2.3 agregando que:

uplift = retención con campaña − retención del grupo de control, y que NUNCA use la tasa de
retención como uplift; si no hay grupo de control, debe pedir ese dato.

al volver a correrlo el agente explica que el 30% no es uplif y pide la tasa del grupo de control, no llama a la herramienta.

la respuesta luego fue:

Para calcular el valor esperado de la campaña de retención, necesitamos usar el concepto de **uplift** (efecto incremental), que mide cuántos clientes se quedaron *gracias* a la campaña, comparado con un grupo de control (clientes que no recibieron la campaña pero tenían las mismas características).

Me diste que el **30% se quedó**, pero esa es la tasa de retención total con la campaña, **no** el uplift.

Para poder usar la herramienta y calcular el resultado correctamente, por favor dime:
* **¿Cuál fue la tasa de retención del grupo de control (sin campaña)?**







*¿en qué trampa cayó el agente y qué cambiaron para arreglarlo?*
